# Exercise 13 : Data Formats and Tools

Count the 2024 road deaths per road-user category, driver (`DRIV`), passenger (`PAS`), pedestrian
(`PED`) and unknown (`UNK`), and the number of countries that report each category, with pandas,
DuckDB and Polars. Use the same selection in every tool and sort the rows by category code.

Read `datasets/eurostat-road-safety-README.md` first. `eurostat-road-deaths-2023-2024.csv` is an
unchanged Eurostat extract of 12,254 rows, retrieved on 10 October 2026. Each row holds one value
for a combination of unit, sex, age, road-user category, country and year. Missing data are absent
rows ; no value is empty.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import polars as pl
import pyarrow.parquet as pq

DATA = Path.cwd().parents[1] / 'datasets'

## Step 1 : Typed load

Read the file with `pd.read_csv()`. Read only these columns with `usecols` :

- `LAST UPDATE`, parsed with `parse_dates` and `date_format='%d/%m/%y %H:%M:%S'`
- The codes `sex`, `age`, `unit`, `pers_cat` and `geo`, with the dtype `category`
- `TIME_PERIOD` and `OBS_VALUE`, with the nullable dtype `Int64`
- `OBS_FLAG`, with the dtype `str`

Rename `LAST UPDATE`, `TIME_PERIOD`, `OBS_VALUE` and `OBS_FLAG` to `last_update`, `year`, `value`
and `flag`. Print the dtypes and the first release time : 25 August 2026, 23:00. Check that no
value is missing and that no combination of the five codes and the year occurs twice.

In [ ]:
# Your code here


## Step 2 : pandas

Select the rows with :

- `unit == 'NR'`, the counts
- `sex == 'T'` and `age == 'TOTAL'`
- The year 2024
- Countries only, without `EU27_2020`

Leave out the road-user category `TOTAL`. Group by `pers_cat` with `observed=True`, sum the values
as `deaths` and count the rows as `countries`.

Check : 126 rows ; `DRIV` 14,078 deaths in 31 countries, `PAS` 3,007 in 31, `PED` 3,706 in 31 and
`UNK` 84 in 33. Check that the categories add up to the 20,875 deaths of the `TOTAL` rows in the
same selection. Why does `UNK` have 33 countries and the other categories 31?

In [ ]:
# Your code here


## Step 3 : Parquet

Write the loaded DataFrame to `deaths.parquet` with `index=False` and read it back. Compare the
dtypes before and after. Print the stored schema with `pyarrow.parquet.read_schema()` : how does
Parquet store the categories? Read only the columns `pers_cat`, `geo`, `year` and `value`.

In [ ]:
# Your code here


## Step 4 : DuckDB

Repeat the count with DuckDB : `con = duckdb.connect()` and a query
`FROM read_parquet('deaths.parquet')` with the same filters, `GROUP BY pers_cat` and
`ORDER BY pers_cat`. Name the sum `deaths` and the count `countries`, and fetch the result with
`.df()`. Are the deaths and countries the same as in pandas?

In [ ]:
# Your code here


## Step 5 : Polars

Repeat it with a lazy query : `pl.scan_parquet('deaths.parquet')`, expressions in `.filter()`,
`.group_by('pers_cat')`, `.agg()` with `pl.col('value').sum()` and `pl.len()`, and
`.sort('pers_cat')`. Print `.explain()` and find the Parquet scan, the selected columns and the
filter. Run the query with `.collect()` and compare the result with pandas.

In [ ]:
# Your code here


## Step 6 : Reflection

Write 5–8 sentences in the answer cell. Explain the selection, why the `TOTAL` rows and
`EU27_2020` are left out, and why an absent row is not a zero. Which tool would you choose for
interactive inspection, and which for repeated queries over a large Parquet file? Mention column
selection and memory, and the difference between Arrow and Parquet.

Your answer here.

## If time permits :

- Run `demos/13-data-formats-and-tools/demo08-numba-optional.py` after
  `uv sync --frozen --all-groups`. Check that the three results agree before you compare the timings